# FDA drug recall Q&A (RAG)
Small project to see how well retrieval plus a small open model can answer questions about FDA drug recalls and point back to the recall number.

openFDA enforcement data -> one chunk per recall -> Chroma -> top-k retrieval -> Qwen 2.5 3B -> test on 20 questions

## 1. Setup
Run this on a Colab GPU runtime.

In [ ]:
!pip install -q chromadb transformers accelerate requests pandas

In [ ]:
import json, random, re
import requests, pandas as pd
import chromadb
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
random.seed(42)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
lm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

def generate(system, user, max_tokens=400):
    chat = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)  # greedy so runs are repeatable
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Setup OK, running on:", lm.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Setup OK, running on: cuda:0


## 2. Get the data
Pulling recalls from the openFDA drug enforcement endpoint. I keep the first 100 that have every field I need and a unique recall number.

In [ ]:
N_RECORDS = 100
url = "https://api.fda.gov/drug/enforcement.json"
params = {"search": "report_date:[20240101 TO 20251231]", "limit": 200}
resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()
results = resp.json()["results"]
print("Fetched", len(results), "records")

KEEP = ["recall_number", "product_description", "recalling_firm", "reason_for_recall",
        "classification", "status", "distribution_pattern", "recall_initiation_date"]
rows, seen = [], set()
for r in results:
    rn = r.get("recall_number")
    if not rn or rn in seen:
        continue
    if not all(r.get(k) for k in KEEP):
        continue
    seen.add(rn)
    rows.append({k: r[k] for k in KEEP})
    if len(rows) == N_RECORDS:
        break

df = pd.DataFrame(rows)
df.to_json("recalls.json", orient="records", indent=2)
print("Kept", len(df), "complete, unique records")
df.head(3)

Fetched 200 records
Kept 100 complete, unique records


,recall_number,product_description,recalling_firm,reason_for_recall,classification,status,distribution_pattern,recall_initiation_date
0,D-0115-2026,"Semaglutide Injection, 10 mg/4 mL (2.5 mg/mL),...",ProRx LLC,Lack of Assurance of Sterility,Class II,Ongoing,TX and UT,20251015
1,D-0572-2024,"Little Moon Essentials, Crampy Belly Rub (Camp...",Little Moon Essentials LLC,CGMP deviations,Class II,Ongoing,"Nationwide USA and Ontario, Canada (2 retailers)",20240620
2,D-0179-2024,"Enoxaparin Sodium Injection, USP, 150MG/ML Sin...",CARDINAL HEALTHCARE,CGMP Deviations: Products were exposed to temp...,Class II,Terminated,Nationwide USA,20220126


## 3. Chunks and embeddings
Each recall is short, so one record is one chunk. Chroma's default embedding model (all-MiniLM-L6-v2) runs locally.

In [ ]:
def to_text(r):
    return (f"Recall {r['recall_number']}. Product: {r['product_description']}. "
            f"Recalling firm: {r['recalling_firm']}. Reason: {r['reason_for_recall']}. "
            f"Classification: {r['classification']}. Status: {r['status']}. "
            f"Distribution: {r['distribution_pattern']}. Initiated: {r['recall_initiation_date']}.")

chroma = chromadb.Client()
try:
    chroma.delete_collection("recalls")
except Exception:
    pass
col = chroma.create_collection("recalls")
col.add(
    ids=df["recall_number"].tolist(),
    documents=[to_text(r) for r in rows],
    metadatas=[{"recalling_firm": r["recalling_firm"], "classification": r["classification"]} for r in rows],
)
print("Indexed", col.count(), "recalls")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 10.2MiB/s]


Indexed 100 recalls


## 4. RAG function
Pull the top-k records, put them in the prompt, and tell the model to answer only from them and cite the recall number.

In [ ]:
SYSTEM = (
    "You answer questions about FDA drug recalls using ONLY the recall records provided. "
    "Cite the recall number in square brackets after each fact, like [D-1234-2025]. "
    "If the records do not contain the answer, say you cannot find it in the provided records. "
    "Keep answers short."
)

def retrieve(question, k=5):
    out = col.query(query_texts=[question], n_results=k)
    return out["ids"][0], out["documents"][0]

def ask(question, k=5):
    ids, docs = retrieve(question, k)
    context = "\n\n".join(docs)
    answer = generate(SYSTEM, f"Recall records:\n{context}\n\nQuestion: {question}")
    return answer, ids

answer, ids = ask("Which recalls were Class I and why?")
print(answer)
print("\nRetrieved:", ids)

None of the recalls listed are Class I.

Retrieved: ['D-0447-2024', 'D-0411-2024', 'D-0563-2025', 'D-0668-2025', 'D-0374-2024']


In [ ]:
for q in ["Which companies recalled products for contamination?",
          "Are there any recalls related to labeling errors?",
          "What was recalled for failing dissolution specifications?"]:
    a, ids = ask(q)
    print("Q:", q); print(a); print("Sources:", ids); print("-" * 60)

Q: Which companies recalled products for contamination?
Specialty Process Labs LLC [D-0447-2024], DermaRite Industries, LLC [D-0168-2026], Thea Pharma, Inc. [D-0374-2024], Private Label Skin Care Inc. [D-0668-2025], Supercore Products Group Inc. [D-0004-2025]
Sources: ['D-0447-2024', 'D-0168-2026', 'D-0374-2024', 'D-0668-2025', 'D-0004-2025']
------------------------------------------------------------
Q: Are there any recalls related to labeling errors?
No
Sources: ['D-0447-2024', 'D-0411-2024', 'D-0668-2025', 'D-0563-2025', 'D-0350-2025']
------------------------------------------------------------
Q: What was recalled for failing dissolution specifications?
Atorvastatin Calcium Tablets USP, 80 mg [D-0020-2026], glipiZIDE, Extended-Release Tablets, 2.5 mg [D-0202-2025], Budesonide Extended-Release Tablets, 9 mg [D-0276-2024]
Sources: ['D-0020-2026', 'D-0202-2025', 'D-0586-2025', 'D-0276-2024', 'D-0217-2026']
------------------------------------------------------------


## 5. Evaluation
20 questions built from the records themselves so I know the right answer (7 classification, 7 firm, 6 reason). I check retrieval hit rate, answer correctness and citation rate at top-3 and top-5.

Correctness is plain token matching against the expected answer. My first try used an LLM as the judge and it gave 75%, which was wrong, so I switched to rules.

The questions include the product and firm names, which makes retrieval easier than it would be with real user questions. The 100% hit rate is optimistic.

In [ ]:
def short(p, n=90):
    p = re.sub(r"\s+", " ", p).strip()
    return p[:n]

sample = df.sample(20, random_state=42).reset_index(drop=True)
tests = []
for i, r in sample.iterrows():
    prod, firm = short(r["product_description"]), r["recalling_firm"]
    if i < 7:
        q = f"What was the recall classification for {prod}, recalled by {firm}?"
        exp = r["classification"]
    elif i < 14:
        q = f"Which company recalled this product: {prod}?"
        exp = firm
    else:
        q = f"Why was {prod}, from {firm}, recalled?"
        exp = r["reason_for_recall"]
    tests.append({"question": q, "expected": exp, "recall_number": r["recall_number"]})

json.dump(tests, open("questions.json", "w"), indent=2)
print(len(tests), "test questions saved")
tests[0]

20 test questions saved


{'question': 'What was the recall classification for Levothyroxine Sodium Tablets, USP, 88 mcg (0.088 mg), 1000 bottles, Rx Only, Manufactured , recalled by ACCORD HEALTHCARE, INC.?',
 'expected': 'Class II',
 'recall_number': 'D-0396-2025'}

In [ ]:
import string
STOP = {"the","a","an","of","and","or","to","for","in","on","by","was","were","is","are"}

def toks(s):
    s = s.lower().translate(str.maketrans("", "", string.punctuation))
    return [w for w in s.split() if w not in STOP]

def is_correct(kind, expected, answer):
    e, a = toks(expected), set(toks(answer))
    if not e:
        return False
    recall = sum(w in a for w in e) / len(e)
    return recall == 1.0 if kind in ("classification", "firm") else recall >= 0.7

def kind_of(i):
    return "classification" if i < 7 else ("firm" if i < 14 else "reason")

def evaluate(k):
    rows_out = []
    for i, t in enumerate(tests):
        answer, ids = ask(t["question"], k)
        kind = kind_of(i)
        rows_out.append({
            "type": kind,
            "question": t["question"],
            "hit": t["recall_number"] in ids,
            "correct": is_correct(kind, t["expected"], answer),
            "cited_retrieved": any(r in answer for r in ids),
            "answer": answer,
            "expected": t["expected"],
        })
    return pd.DataFrame(rows_out)

summary, details = {}, {}
for k in (3, 5):
    d = evaluate(k)
    details[k] = d
    summary[f"top-{k}"] = {
        "retrieval hit rate %": round(d["hit"].mean() * 100),
        "answer correct %": round(d["correct"].mean() * 100),
        "citation rate %": round(d["cited_retrieved"].mean() * 100),
    }
print(pd.DataFrame(summary))
print()
print(details[5].groupby("type")[["hit", "correct"]].mean().round(2))

                      top-3  top-5
retrieval hit rate %    100    100
answer correct %         90    100
citation rate %          90     95

                hit  correct
type                        
classification  1.0      1.0
firm            1.0      1.0
reason          1.0      1.0


## 6. Failures

In [ ]:
d = details[5]
fails = d[~d["correct"]]
print(len(fails), "incorrect answers at top-5")
for _, r in fails.head(5).iterrows():
    print("Q:", r["question"]); print("Expected:", r["expected"]); print("Got:", r["answer"]); print("Retrieval hit:", r["hit"]); print("-" * 60)

0 incorrect answers at top-5


No wrong answers at top-5 on the test set, but the demo in section 4 shows the real problem: "Which recalls were Class I and why?" came back with "None of the recalls listed are Class I." Semantic search can't filter on a field, so questions like this fail. Next step is to use the classification metadata as a Chroma filter and add a small router that decides when to filter and when to search.

## 7. Save results

In [ ]:
pd.DataFrame(summary).to_csv("eval_results.csv")
print(pd.DataFrame(summary))

                      top-3  top-5
retrieval hit rate %    100    100
answer correct %         90    100
citation rate %          90     95
